# Colour Brings People Together
### Part C · Creating an evidence-based colour palette (45 min)

<img src="https://raw.githubusercontent.com/akvilesinkeviciute/colour-brings-people-together/v1.0/images/serandu_example.png" alt="Serandu in OtjiHimba: the AI's estimate, a brown, against speakers' best example, a wine red, ΔE₀₀ 25.4 apart" width="420">

Part B went from colours to names. Now go the other way: give your names to the AI, ask which colour each one means, check its answers against the Colour Spinner, and fix the palette for each audience, with evidence.

*Why: in the Colour in Translation benchmark (Mylonas et al., CHI 2026), a current large language model's colours were far from speakers' focal colours, because it mapped names onto the same prototypes in every language.*

Run each cell with **Shift + Enter**. Change only the grey forms and the menus.

<details><summary><b>Why a different palette for each language?</b></summary>

Because colour meaning differs between languages: the palette is designed for the people reading it, not for one 'universal' viewer.
</details>

In [ ]:
#@title Run me first: load the masterclass toolkit
#@markdown Takes about 30 seconds the first time. You're ready when it says **Toolkit ready ✓**.
TOOLKIT_URL = "https://raw.githubusercontent.com/akvilesinkeviciute/colour-brings-people-together/v1.0/colour_masterclass.py"
import os, importlib, urllib.request
try:                                   # always fetch the current version
    urllib.request.urlretrieve(TOOLKIT_URL, "colour_masterclass.py")
except Exception as e:
    if not os.path.exists("colour_masterclass.py"):
        raise RuntimeError("Could not download the toolkit. Check your internet connection, or upload "
                           "colour_masterclass.py in Colab's Files panel (folder icon, left).") from e
    print("Couldn't refresh the toolkit, so using the copy already here.")
import colour_masterclass
importlib.reload(colour_masterclass)
from colour_masterclass import *

## 1 · Your names from Part B

Run the cell and upload `partB_results.csv`. Lost it? Choose *Type them in*.

In [ ]:
#@title Load your names
NamesFrom = "Upload partB_results.csv"  #@param ["Upload partB_results.csv", "Type them in"]
#@markdown Run this cell and upload your Part B results. *Type them in* only if you lost the file.
Backend = "Colour Spinner API"   # facilitators: switch to "Copied from the browser" only if the Spinner can't be reached
names_first(NamesFrom)                    # only with "Type them in": shows boxes to fill in first
names = load_names(NamesFrom)
LANGUAGES = list(dict.fromkeys(names.language))
show_names(names)

## 2 · Ask your AI for the colour of each name

Use the same AI as in Part B. It's asked for each name's **best example**: its most typical colour.

**Try asking twice.** For ChatGPT you don't need an account, and it's better not to log in, so that it has no memory of you. Paste its first reply into the box; then open a **new chat**, ask again, and compare the two replies yourselves: does it give the same answers? Inconsistent answers are worth discussing too.

In [ ]:
#@title The prompt for your AI
lines = "\n".join(numbered_items(names, "colour"))
himba = "\n(OtjiHimba is spoken by the Himba of northern Namibia.)" if "OtjiHimba" in LANGUAGES else ""
PROMPT_C = f"""You are helping with a cross-cultural colour naming study.
For each numbered colour name below, give the colour, as a hex code, that native speakers of that language would consider the best example of the name: its most typical instance.{himba}
If you're not certain, give your best estimate: we will compare it with empirical data.

{lines}

Reply with one line per number, in this format and nothing else:
1. #RRGGBB
2. …"""
show_prompt(PROMPT_C, PASTE, rows=names, kind="colour")

In [ ]:
#@title Read the AI's colours
pal = read_ai_colours(collected_answers(), names)       # the colour codes read from the pasted reply

## 3 · Does the AI's colour mean the same to speakers?

Two checks per name:
- **Distance:** ΔE₀₀ between the AI's colour and speakers' best example. Above **10**, they're different colours, as in Part B.
- **Round trip:** what speakers call the AI's colour. Another name means the AI's colour says something else.

In [ ]:
#@title Compare the AI's colours with the speakers' best examples
ReplaceAbove = DELTA_E_THRESHOLD              # ΔE₀₀ 10, as in Part B

show(small("Looking up every name's best example and what speakers call the AI's colours; this takes a minute, please wait…"))
spinner = connect(Backend)
pal = add_best_examples(pal, spinner)                   # the speakers' best example of each name
pal["de00"] = [de00(ai, best) if isinstance(ai, str) and isinstance(best, str) else np.nan
               for ai, best in zip(pal.ai_hex, pal.spinner_hex)]
pal["rt_name"], pal["rt_conf"] = None, np.nan
for k, row in pal.iterrows():                           # round trip: what speakers call the AI's colour
    if isinstance(row.ai_hex, str):
        namer = safe(spinner.name, row.ai_hex, row.language)
        if namer:
            pal.at[k, "rt_name"], pal.at[k, "rt_conf"] = namer["name"], namer["confidence"]
pal = pal.sort_values(["language", "de00"], ascending=[True, False]).reset_index(drop=True)
show_ai_grid(pal, LANGUAGES, ReplaceAbove)
error_summary(pal, LANGUAGES, ReplaceAbove)

**Where is the AI off, and in which direction?** Each panel: the names for one test colour, on the CIELAB a*–b* plane, seen from above. Outlined shape: the AI's colour; filled: speakers' best example. Lines all pointing one way suggest a systematic bias, for instance towards English focal colours.

In [ ]:
#@title Where the AI is off
plot_ai_vs_speakers(pal, LANGUAGES, ReplaceAbove)

## 4 · Optimise the palette, in two steps, then a check

Each step shows the palette in the same grid; after Step 2, it's your final palette.

### Step 1 · Optimise for meaning: use what speakers mean

The AI's colour counts as wrong if it's too far from speakers' best example of the name (above ΔE₀₀ 10), or if speakers would call it by another name (the round trip). Wherever it's wrong, it's replaced by speakers' best example; otherwise it's kept.

In [ ]:
#@title Step 1 · Optimise for meaning
pal_grounded = ground_palette(pal, "Either")      # wrong if too far from the best example, or speakers would use another name
show_step_palette(pal_grounded, "ai", LANGUAGES, "Step 1 · Your palette, optimised for meaning", "optimised · meaning",
                  unchanged_label="kept",
                  note="Left: the AI's colour; right: after this step. Where the AI's colour was wrong, it is now speakers' best example of the name.")

### Step 2 · Give every test colour its own name

Two test colours with the same name (6 and 14 are both *burou* in OtjiHimba) would be confused in conversation. One of them takes the closest Thesaurus synonym whose best example stands apart from your other colours. Then each colour goes with the test colour it's closest to (ΔE₀₀): the darker blue with the darker test colour, the lighter with the lighter. Every colour stays the best example of its name. With OtjiHimba's seven terms, the nearest names may be taken, so the new name can mean something rather different: communicable, but at a cost.

In [ ]:
#@title Step 2 · Rename shared names
pal_named = name_palette(pal_grounded, spinner, LANGUAGES)

### Step 3 · Check: how your palette looks to other viewers

Choose a kind of vision. On the left, your final colours with typical colour vision; on the right, the same colours as a viewer with that vision sees them. Colours that look alike on the right need a second cue, such as a pattern, a label or an icon.

In [ ]:
#@title Step 3 · Check { run: "auto" }
Vision = "Deuteranopia"  #@param ["Protanopia", "Deuteranopia", "Tritanopia", "Age 40", "Age 60", "Age 80"]
#@markdown **Vision**: whose eyes to check the palette with.
pal = check_vision(pal_named, LANGUAGES, Vision)

## 5 · See the palette in use

Your final palette on swatches, charts or a world map (example data), next to another kind of vision. **Colours:** your test colours, or all 17 SDG colours. Every preview you make is saved for the download.

In [ ]:
#@title Preview
#@markdown Choose a language, which colours, a kind of design and a kind of vision.
live_controls({"language": ("Language", LANGUAGES, my_language(names)),
               "colours":  ("Colours", ["Your test colours", "All 17 SDG colours"], "Your test colours"),
               "design":   ("Design", ["Colour swatches", "Bar chart", "Donut chart", "Line chart", "Scatter plot", "Map (world)"], "Bar chart"),
               "vision":   ("Vision", VISIONS, "Deuteranopia")},
              lambda language, colours, design, vision: preview_applications(pal, language, vision, kind=design, colours=colours,
                  path=f"preview_{language.replace(' ', '_')}_{design.split(' ')[0].lower()}{'_all17' if colours.startswith('All') else ''}_{vision.replace(' ', '_').lower()}.png"))

## 6 · Your colours in the full SDG palette

The UN's 17 SDG colours, with your test colours replaced by your final colours, and the same palette in colour space, as in Part B: arrows show where each test colour moved. Choose a **Language** and a kind of **Vision**.

*Only your test colours changed, and each goal also has a number, an icon and a name: an example of what evidence can change, not a proposal for the UN's colours.*

In [ ]:
#@title Your colours in the full SDG palette
#@markdown Choose a language and a kind of vision.
live_controls({"language": ("Language", [l for l in LANGUAGES if l in set(pal.language)], my_language(names)),
               "vision":   ("Vision", VISIONS, "Typical")},
              lambda language, vision: show_final_palette(pal, language, vision))

## 7 · Export your palette

One zip with everything above: your palette by test colour and language (a picture, and a table with the evidence), the full SDG palette with your final colours for each language (a picture and a table), and your previews.

In [ ]:
#@title Download your palette
export_palette(pal, LANGUAGES)

## 8 · Your most striking finding

For each language, the name where the AI's colour was furthest from what speakers mean. Choose a **Language**, and bring your finding to the plenary.

In [ ]:
#@title Finding to share
live_controls({"language": ("Language", [l for l in LANGUAGES if l in set(pal.language)], my_language(names))},
              lambda language: show_finding(pal_grounded, language))   # the names the AI was asked about

## 9 · Tell us how it went (optional)

In [ ]:
#@title The questionnaire
show_questionnaire()

---
#### Credits and references

**SDG colours:** United Nations, [Sustainable Development Goals communications materials](https://www.un.org/sustainabledevelopment/news/communications-material/); [un.org/sustainabledevelopment](https://www.un.org/sustainabledevelopment). *The content of this publication has not been approved by the United Nations and does not reflect the views of the United Nations or its officials or Member States.*

**Colour naming data and simulations:** the Colour Spinner, [colornaming.net](https://colornaming.net); Sinkeviciute, Ahmed, Stutters, Koliousis & Mylonas (2026), under review. Colour vision deficiencies: Viénot, Brettel & Mollon (1999); ageing: Pokorny, Smith & Lutze (1987).

**AI colour-naming benchmark:** Mylonas, Ahmed, Sinkeviciute & Koliousis, *Colour in Translation*, CHI 2026.

**The *serandu* example:** the AI's estimate from ChatGPT, asked on 26 October 2026; the speakers' best example from the Colour Spinner.

**Maps:** world map from Natural Earth (public domain) and US states map from the US Census Bureau, both via vega-datasets; example values only.

**Colour science:** CIELAB and CIEDE2000 with the [colour-science](https://www.colour-science.org) library.